# Descriptive Statistics & Predictor Readiness Check
## Postoperative Complications After Lung Cancer Surgery — DLCAS-2026

---

**Goals of this notebook:**
1. Load the SPSS (.sav) dataset and explore its structure
2. Describe the distribution of each predictor (mean, median, missing values, outliers)
3. Visualize distributions for continuous and categorical variables
4. Check each variable for issues (missingness, sentinel values, outliers)
5. Produce a final **readiness summary table** — which variables are ready for modeling and which need attention

**Predictor groups covered:**
- Demographics & body composition
- Functional status & risk scores (ASA, ECOG, FEV1%, DLCO%, VO2max)
- Comorbidities
- Imaging & pre-operative staging (TNM)
- Neoadjuvant treatment
- Procedure / surgical factors
- COVID-19 pre-operative status

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings

try:
    import pyreadstat
except ImportError:
    raise ImportError("Install pyreadstat:  pip install pyreadstat")

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.2f}'.format)
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
sns.set_palette('Blues_d')

# ── CONFIG ────────────────────────────────────────────────────
# NOTE: source registry data is not included in this public repo.
# Path below is a relative placeholder — point it at your own copy.
FILE_PATH = "../../data/dlca_lung_raw.sav"   # raw DLSA registry export (not in repo)

# Sentinel values used in this registry for 'unknown'
BINARY_SENTINELS  = [9]
NUMERIC_SENTINELS = [999, 9999, -99, -999, 99.9]

# Outcome variable mapping
OUTCOMES = {
    'compl':         'Any complication',
    'compluchtweg':  'Airway',
    'compmechan':    'Mechanical airway',
    'comptrombo':    'Thromboembolism',
    'compcardio':    'Cardiac',
    'compinfect':    'Infection',
    'comprespi':     'Respiratory',
    'compuro':       'Urological',
    'compproc':      'Procedure-related',
    'compalg':       'General',
    'compbl':        'Post-op bleeding',
    'reinterv':      'Re-intervention',
    'status':        '30-day mortality',
}

# Predictor groups (column names after lowercasing)
PREDICTORS = {
    'Demographics': [
        'leeftijd', 'geslacht', 'lengte', 'gewicht', 'bmi', 'roken', 'alcohol'
    ],
    'Functional Status': [
        'asascore', 'ecog', 'prelongf', 'prelongf1', 'prelongfvo2max'
    ],
    'Comorbidities': [
        'compulmobstr', 'compulmfibr', 'compulmtranspl',
        'comap', 'commyo', 'comcabg', 'comhartfaal', 'comaf', 'comcarritme',
        'comhypert', 'comperif', 'comcva', 'comparalys', 'comdem',
        'comnier', 'comdiam', 'commalig', 'comlever', 'comgiulcus', 'combind', 'comhiv'
    ],
    'Imaging & Staging (pre-op)': [
        'ctscan', 'pet', 'tumorpetpos', 'tumcentraal',
        'prescoretnm8ct', 'prescoretnm8cn', 'prescoretnm8cm',
        'prescoretnm9ct', 'prescoretnm9cn', 'prescoretnm9cm'
    ],
    'Neoadjuvant Treatment': [
        'neoadj', 'inductccrtx', 'inductctx', 'inductrtx', 'inductbiol', 'inductimmuno'
    ],
    'Procedure': [
        'reggroep', 'procok', 'toegang', 'zijde', 'sleeve',
        'veraanv', 'verbronc', 'verthor', 'verdiaf', 'verperi', 'vercard', 'vervaat',
        'opnduur', 'compbloed', 'compbloedver', 'transfusie', 'peropcompl'
    ],
    'COVID (pre-op)': ['covpredefinitief']
}

CONTINUOUS = [
    'leeftijd', 'lengte', 'gewicht', 'bmi',
    'prelongf', 'prelongf1', 'prelongfvo2max',
    'opnduur', 'compbloedver'
]

---
## 1. Load Data & Explore Structure

In [ ]:
df, meta = pyreadstat.read_sav(FILE_PATH)

# Normalize all column names to lowercase
df.columns = [c.lower() for c in df.columns]

# Lowercase metadata keys too so lookups are consistent
meta_labels    = {k.lower(): v for k, v in meta.column_names_to_labels.items()}
meta_val_labels = {k.lower(): v for k, v in meta.variable_value_labels.items()}

print(f"Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"\nAll columns ({len(df.columns)} total):")
print(f"{'#':<5} {'Column':<40} {'SPSS Label'}")
print("-" * 80)
for i, col in enumerate(df.columns):
    lbl = meta_labels.get(col, '')
    print(f"{i+1:<5} {col:<40} {lbl}")

In [ ]:
# Check which expected predictors are present vs missing in the dataset
print("Predictor coverage check:")
print("=" * 60)
all_expected = []
for grp, cols in PREDICTORS.items():
    found   = [c for c in cols if c in df.columns]
    missing = [c for c in cols if c not in df.columns]
    all_expected.extend(cols)
    print(f"\n  {grp}: {len(found)}/{len(cols)} found")
    if missing:
        print(f"    NOT FOUND: {missing}")

total_found = len([c for c in all_expected if c in df.columns])
print(f"\nTotal: {total_found}/{len(all_expected)} predictor columns found")
print("\nNOTE: columns marked NOT FOUND may use different names in this SPSS file.")
print("Cross-reference the column list above to identify the correct names.")

In [ ]:
# Clean sentinel values (registry codes for 'unknown') → NaN
# Binary flags (0/1): 9 = unknown
# Numeric/continuous: 999, -99, 99.9 = unknown

all_predictors_flat = [c for grp in PREDICTORS.values() for c in grp if c in df.columns]
n_cleaned_total = 0

for col in all_predictors_flat:
    series = pd.to_numeric(df[col], errors='coerce')
    unique_vals = set(series.dropna().unique())

    # Binary-like: only values in {0, 1, 9}
    if unique_vals.issubset({0.0, 1.0, 9.0}):
        n = series.isin(BINARY_SENTINELS).sum()
        if n > 0:
            df[col] = series.replace(BINARY_SENTINELS, np.nan)
            n_cleaned_total += n
            print(f"  {col:<30} {n} sentinel (9) → NaN")
    else:
        n = series.isin(NUMERIC_SENTINELS).sum()
        if n > 0:
            df[col] = series.replace(NUMERIC_SENTINELS, np.nan)
            n_cleaned_total += n
            print(f"  {col:<30} {n} numeric sentinels → NaN")

print(f"\nTotal sentinel values cleaned: {n_cleaned_total}")

In [ ]:
# --- Age at surgery ---
if 'leeftijd' in df.columns:
    df['leeftijd'] = pd.to_numeric(df['leeftijd'], errors='coerce')
    valid = df['leeftijd'].notna().sum()
    print(f"leeftijd (age): {valid} valid values")
    print(f"  Range: {df['leeftijd'].min():.0f} - {df['leeftijd'].max():.0f} years")
    print(f"  Mean:  {df['leeftijd'].mean():.1f} years")
    bad = ((df['leeftijd'] < 18) | (df['leeftijd'] > 100)).sum()
    if bad > 0:
        print(f"  WARNING: {bad} implausible ages (<18 or >100) — check source data")
elif 'gebdat' in df.columns and 'datok' in df.columns:
    datok_dt  = pd.to_datetime(df['datok'],  errors='coerce', dayfirst=True)
    gebdat_dt = pd.to_datetime(df['gebdat'], errors='coerce', dayfirst=True)
    df['leeftijd'] = (datok_dt - gebdat_dt).dt.days // 365
    print(f"leeftijd derived from gebdat + datok: {df['leeftijd'].notna().sum()} valid values")
else:
    print("WARNING: leeftijd not found and gebdat/datok missing — age unavailable")

# --- BMI ---
if 'bmi' not in df.columns:
    if 'gewicht' in df.columns and 'lengte' in df.columns:
        gew  = pd.to_numeric(df['gewicht'], errors='coerce')
        len_ = pd.to_numeric(df['lengte'],  errors='coerce')
        df['bmi'] = gew / ((len_ / 100) ** 2)
        print(f"\nBMI computed: {df['bmi'].notna().sum()} valid values")
        print(f"  Range: {df['bmi'].min():.1f} - {df['bmi'].max():.1f} kg/m\u00b2")
    else:
        print("\nWARNING: lengte/gewicht not found — BMI unavailable")
else:
    df['bmi'] = pd.to_numeric(df['bmi'], errors='coerce')
    print(f"\nBMI (from dataset): {df['bmi'].notna().sum()} valid values")
    print(f"  Range: {df['bmi'].min():.1f} - {df['bmi'].max():.1f} kg/m\u00b2")

print(f"\nFinal dataset ready: {df.shape[0]} patients, {df.shape[1]} columns")

---
## 2. Target Variables Overview

Before describing predictors, we check the distribution of outcome variables.
These are **labels** — not predictors — and must not be fed into any model.

Lung-specific complications include airway complications (anastomotic/bronchial),
respiratory failure, cardiac events, thromboembolism, and re-interventions.

In [ ]:
comp_present = [c for c in OUTCOMES if c in df.columns]

# Denominator: total cohort size (N patients), used consistently for every
# outcome's "% of total" — NOT each column's own non-missing count.
# Using per-column notna() as the denominator was inflating subtype rates
# (e.g. Airway, Cardiac) because those columns are sparsely populated.
total_n = len(df)

print("=" * 58)
print(f"{'Outcome':<28} {'N events':>9} {'% of total':>12}")
print("=" * 58)
for col in comp_present:
    series = pd.to_numeric(df[col], errors='coerce')
    n      = (series == 1).sum()
    pct    = n / total_n * 100 if total_n > 0 else 0
    flag   = "   <50 events" if n < 50 else ("   <100 events" if n < 100 else "")
    print(f"{OUTCOMES[col]:<28} {int(n):>9} {pct:>11.1f}%{flag}")
print("=" * 58)
print(f"\nDenominator used for all '% of total' values: N = {total_n} (total cohort)")

missing_outcomes = [c for c in OUTCOMES if c not in df.columns]
if missing_outcomes:
    print(f"\nOutcome columns NOT found in dataset: {missing_outcomes}")

In [ ]:
# Clavien-Dindo severity distribution
clav_cols = [c for c in df.columns if c.startswith('clav')]
print(f"Clavien-Dindo columns found: {clav_cols}")

if clav_cols:
    df['derived_clavien'] = (
        df[clav_cols]
        .apply(pd.to_numeric, errors='coerce')
        .replace({9: np.nan})
        .max(axis=1)
    )
    df['derived_clavien_bin'] = np.where(
        df['derived_clavien'].between(1, 2), 0,
        np.where(df['derived_clavien'].between(3, 7), 1, np.nan)
    )

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    grade_counts = df['derived_clavien'].value_counts(dropna=False).sort_index()
    axes[0].bar([str(x) for x in grade_counts.index], grade_counts.values,
                color='steelblue', edgecolor='white')
    axes[0].set_title('Clavien-Dindo Grade Distribution', fontweight='bold', fontsize=13)
    axes[0].set_xlabel('Grade')
    axes[0].set_ylabel('Number of patients')
    for i, v in enumerate(grade_counts.values):
        axes[0].text(i, v + 2, str(v), ha='center', fontsize=10)

    bin_counts = df['derived_clavien_bin'].value_counts(dropna=False).sort_index()
    label_map  = {0.0: 'Minor (1-2)', 1.0: 'Major (3-7)'}
    pie_labels = [label_map.get(k, 'Missing') for k in bin_counts.index]
    axes[1].pie(bin_counts.values, labels=pie_labels,
                autopct='%1.1f%%', colors=['#88BBDD', '#E07050', '#C0C0C0'],
                startangle=90)
    axes[1].set_title('Minor vs Major Complications', fontweight='bold', fontsize=13)

    plt.tight_layout()
    plt.show()
else:
    print("No clav* columns found — check column names in section 1.")

---
## 3. Missing Data Overview

**Decision rule used in this study:**

| Threshold | Action |
|---|---|
| > 50% missing | Exclude from model |
| 20–50% missing | Include but flag — imputation required |
| < 20% missing | Ready to use |

Note: missing counts here are **after** sentinel cleaning (9/999/−99 → NaN).

In [ ]:
all_predictors = [c for grp in PREDICTORS.values() for c in grp if c in df.columns]

missing = pd.DataFrame({
    'Variable':   all_predictors,
    'N missing':  [df[c].isna().sum() for c in all_predictors],
    'N present':  [df[c].notna().sum() for c in all_predictors],
    '% missing':  [round(df[c].isna().mean() * 100, 1) for c in all_predictors]
}).sort_values('% missing', ascending=False)

def flag_missing(pct):
    if pct > 50:  return '  Exclude'
    elif pct > 20: return '  Impute (flag)'
    else:          return '  OK'

missing['Status'] = missing['% missing'].apply(flag_missing)

print(missing[missing['% missing'] > 0].to_string(index=False))
print(f"\nVariables with 0% missing: {(missing['% missing'] == 0).sum()}")
not_found = len([c for grp in PREDICTORS.values() for c in grp]) - len(all_predictors)
if not_found > 0:
    print(f"Variables not found in dataset: {not_found} (see section 1 coverage check)")

In [ ]:
fig, ax = plt.subplots(figsize=(14, max(6, len(all_predictors) * 0.32)))

miss_plot = missing[missing['% missing'] > 0].sort_values('% missing', ascending=True)
bar_colors = ['#E07050' if p > 50 else '#F0A860' if p > 20 else '#88BBDD'
              for p in miss_plot['% missing']]

bars = ax.barh(miss_plot['Variable'], miss_plot['% missing'],
               color=bar_colors, edgecolor='white')
ax.axvline(50, color='red',    linestyle='--', linewidth=1.5, label='50% threshold (exclude)')
ax.axvline(20, color='orange', linestyle='--', linewidth=1.5, label='20% threshold (flag)')
ax.set_xlabel('% Missing', fontsize=12)
ax.set_title('Missing Data by Predictor Variable', fontweight='bold', fontsize=14)
ax.legend(loc='lower right')

for bar, pct in zip(bars, miss_plot['% missing']):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
            f'{pct:.1f}%', va='center', fontsize=9)

plt.tight_layout()
plt.show()

---
## 4. Continuous Variables

For each continuous predictor we report N, mean, median, std, min, max, % missing,
and outlier count (IQR method: values beyond Q1 − 1.5×IQR or Q3 + 1.5×IQR).

Lung-specific notes:
- **prelongf** = FEV1% predicted (valid range ~20–150; 999 = unknown)
- **prelongf1** = DLCO% predicted
- **prelongfvo2max** = VO2max ml/kg/min (99.9 = unknown)
- **opnduur** = operation duration (minutes)
- **compbloedver** = intra-op blood loss (ml)

In [ ]:
cont_present = [c for c in CONTINUOUS if c in df.columns]

rows = []
for col in cont_present:
    s = pd.to_numeric(df[col], errors='coerce').dropna()
    if len(s) == 0:
        continue
    Q1, Q3 = s.quantile(0.25), s.quantile(0.75)
    IQR = Q3 - Q1
    n_out = ((s < Q1 - 1.5 * IQR) | (s > Q3 + 1.5 * IQR)).sum()
    rows.append({
        'Variable':  col,
        'N':         len(s),
        '% missing': round(df[col].isna().mean() * 100, 1),
        'Mean':      round(s.mean(), 2),
        'Median':    round(s.median(), 2),
        'Std':       round(s.std(), 2),
        'Min':       round(s.min(), 2),
        'Max':       round(s.max(), 2),
        'N outliers (IQR)': int(n_out)
    })

cont_summary = pd.DataFrame(rows)
print(cont_summary.to_string(index=False))

In [ ]:
n_cols = 3
n_rows = int(np.ceil(len(cont_present) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows * 4))
axes = axes.flatten()

for i, col in enumerate(cont_present):
    ax   = axes[i]
    data = pd.to_numeric(df[col], errors='coerce').dropna()
    ax.hist(data, bins=30, color='steelblue', edgecolor='white', alpha=0.85)
    ax.axvline(data.median(), color='red', linestyle='--', linewidth=1.5,
               label=f'Median: {data.median():.1f}')
    ax.set_title(col, fontweight='bold', fontsize=12)
    ax.set_xlabel('Value')
    ax.set_ylabel('Count')
    ax.legend(fontsize=9)
    miss_pct = df[col].isna().mean() * 100
    ax.text(0.98, 0.95, f'Missing: {miss_pct:.1f}%',
            transform=ax.transAxes, ha='right', va='top', fontsize=9, color='gray')

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Distribution of Continuous Predictors', fontweight='bold', fontsize=15, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows * 4))
axes = axes.flatten()

for i, col in enumerate(cont_present):
    ax   = axes[i]
    data = pd.to_numeric(df[col], errors='coerce').dropna()
    ax.boxplot(data, vert=True, patch_artist=True,
               boxprops=dict(facecolor='steelblue', alpha=0.6),
               medianprops=dict(color='red', linewidth=2))
    ax.set_title(col, fontweight='bold', fontsize=12)
    ax.set_ylabel('Value')
    Q1, Q3 = data.quantile(0.25), data.quantile(0.75)
    n_out  = ((data < Q1 - 1.5*(Q3-Q1)) | (data > Q3 + 1.5*(Q3-Q1))).sum()
    ax.text(0.98, 0.95, f'Outliers: {n_out}',
            transform=ax.transAxes, ha='right', va='top',
            fontsize=9, color='darkorange' if n_out > 0 else 'gray')

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Boxplots of Continuous Predictors', fontweight='bold', fontsize=15, y=1.01)
plt.tight_layout()
plt.show()

---
## 5. Binary & Categorical Variables

For binary flags we report: N=1 (cases), % positive (prevalence), % missing.

Flags:
- **Low prevalence (<2%)** — may not provide enough signal for modeling
- **Missing >20%** — imputation required before modeling

In [ ]:
# All predictor variables that are not in the continuous list
binary_cat_vars = [c for grp in PREDICTORS.values() for c in grp
                   if c in df.columns and c not in CONTINUOUS]

rows = []
for col in binary_cat_vars:
    series  = pd.to_numeric(df[col], errors='coerce')
    n_one   = (series == 1).sum()
    n_valid = series.notna().sum()
    pct_one = n_one / n_valid * 100 if n_valid > 0 else 0
    miss    = df[col].isna().mean() * 100
    n_uniq  = series.nunique(dropna=True)
    vtype   = 'binary' if n_uniq <= 2 else 'categorical'
    flag    = ''
    if vtype == 'binary' and pct_one < 2: flag = '  Low prevalence (<2%)'
    if miss > 20: flag += '  Missing >20%'
    rows.append({
        'Variable':   col,
        'Type':       vtype,
        'N (=1)':     int(n_one),
        '% positive': round(pct_one, 1),
        '% missing':  round(miss, 1),
        'N unique':   int(n_uniq),
        'Status':     flag.strip()
    })

bin_summary = pd.DataFrame(rows)
print(bin_summary.to_string(index=False))

In [ ]:
com_vars = [c for c in PREDICTORS['Comorbidities'] if c in df.columns]

com_labels = {
    'compulmobstr':  'COPD/CARA/Emphysema',
    'compulmfibr':   'Pulmonary fibrosis',
    'compulmtranspl':'Post lung transplant',
    'comap':         'Angina',
    'commyo':        'Prior MI',
    'comcabg':       'CABG/PTCA',
    'comhartfaal':   'Heart failure',
    'comaf':         'Atrial fibrillation',
    'comcarritme':   'Other arrhythmia',
    'comhypert':     'Hypertension',
    'comperif':      'Peripheral vascular',
    'comcva':        'Stroke/TIA',
    'comparalys':    'Paralysis',
    'comdem':        'Dementia',
    'comnier':       'Renal disease',
    'comdiam':       'Diabetes',
    'commalig':      'Prior malignancy',
    'comlever':      'Liver disease',
    'comgiulcus':    'GI ulcer',
    'combind':       'Connective tissue',
    'comhiv':        'HIV'
}

prevalences = []
for c in com_vars:
    series  = pd.to_numeric(df[c], errors='coerce')
    n_valid = series.notna().sum()
    if n_valid > 0:
        pct = (series == 1).sum() / n_valid * 100
        prevalences.append((com_labels.get(c, c), pct))

prevalences.sort(key=lambda x: x[1], reverse=True)
names, vals = zip(*prevalences)

fig, ax = plt.subplots(figsize=(13, 7))
bar_colors = ['#E07050' if v < 2 else '#88BBDD' for v in vals]
bars = ax.barh(names, vals, color=bar_colors, edgecolor='white')
ax.axvline(2, color='red', linestyle='--', linewidth=1, alpha=0.6, label='2% threshold')
ax.set_xlabel('Prevalence (%)', fontsize=12)
ax.set_title('Comorbidity Prevalence — Lung Surgery Cohort', fontweight='bold', fontsize=14)
ax.legend()
for bar, val in zip(bars, vals):
    ax.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height() / 2,
            f'{val:.1f}%', va='center', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Categorical variables (ordinal scales, procedure codes, TNM stages)
CAT_VARS = [
    'reggroep', 'procok', 'toegang', 'zijde', 'sleeve', 'compbloed',
    'roken', 'alcohol', 'asascore', 'ecog',
    'prescoretnm8ct', 'prescoretnm8cn', 'prescoretnm8cm',
    'prescoretnm9ct', 'prescoretnm9cn', 'prescoretnm9cm'
]
cat_present = [c for c in CAT_VARS if c in df.columns]

n_cols = 3
n_rows = int(np.ceil(len(cat_present) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, n_rows * 4))
axes = axes.flatten()

for i, col in enumerate(cat_present):
    ax = axes[i]
    vc = df[col].value_counts(dropna=False).head(12)
    # Use SPSS value labels if available
    val_lbl = meta_val_labels.get(col, {})
    disp_labels = [val_lbl.get(k, str(k)) if not pd.isna(k) else 'Missing' for k in vc.index]
    colors_bar  = ['#C0C0C0' if pd.isna(k) else 'steelblue' for k in vc.index]
    ax.bar(disp_labels, vc.values, color=colors_bar, edgecolor='white')
    ax.set_title(col, fontweight='bold', fontsize=12)
    ax.set_ylabel('Count')
    plt.setp(ax.get_xticklabels(), rotation=35, ha='right', fontsize=8)
    miss = df[col].isna().mean() * 100
    ax.text(0.98, 0.95, f'Missing: {miss:.1f}%',
            transform=ax.transAxes, ha='right', va='top', fontsize=9, color='gray')

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Distribution of Categorical Predictors', fontweight='bold', fontsize=15, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# SPSS value labels reference for key categorical variables
KEY_VARS = [
    'geslacht', 'roken', 'alcohol', 'asascore', 'ecog',
    'reggroep', 'procok', 'toegang', 'zijde', 'sleeve',
    'compbloed', 'transfusie',
    'prescoretnm8ct', 'prescoretnm8cn', 'prescoretnm8cm'
]

print("SPSS Value Labels for Key Variables")
print("=" * 60)
for col in KEY_VARS:
    if col not in df.columns:
        print(f"\n  {col}  [NOT FOUND IN DATASET]")
        continue
    spss_lbl  = meta_labels.get(col, '')
    val_dict  = meta_val_labels.get(col, {})
    counts    = df[col].value_counts(dropna=False)
    print(f"\n  {col}  [{spss_lbl}]  (N unique: {df[col].nunique(dropna=True)})")
    if val_dict:
        for code, label in sorted(val_dict.items(), key=lambda x: str(x[0])):
            n = counts.get(code, 0)
            print(f"    {str(code):<8} = {label:<35} (N={n})")
    else:
        top = counts.head(8)
        for val, n in top.items():
            print(f"    {str(val):<8}   (N={n})  [no label in metadata]")

---
## 6. Univariate Association with Complication Outcome

Exploratory check: does each predictor show a signal towards `compl` (any complication)?

- **Continuous**: t-test comparing means between complication / no-complication groups
- **Binary**: chi-square comparing complication rates

A non-significant result here **does not** mean the variable should be excluded —
it may still contribute in a multivariate model. This is purely exploratory.

In [ ]:
if 'compl' not in df.columns:
    print("'compl' column not found — skipping univariate association.")
else:
    outcome = pd.to_numeric(df['compl'], errors='coerce')

    print("CONTINUOUS VARIABLES vs compl (any complication)")
    print("=" * 68)
    print(f"{'Variable':<22} {'Mean (no compl)':>16} {'Mean (compl)':>14} {'p-value':>10}")
    print("=" * 68)

    for col in cont_present:
        data     = pd.to_numeric(df[col], errors='coerce')
        no_comp  = data[outcome == 0].dropna()
        yes_comp = data[outcome == 1].dropna()
        if len(no_comp) > 5 and len(yes_comp) > 5:
            _, p = stats.ttest_ind(no_comp, yes_comp)
            sig  = ' *' if p < 0.05 else ''
            print(f"{col:<22} {no_comp.mean():>16.2f} {yes_comp.mean():>14.2f} {p:>10.4f}{sig}")

    print("\n* p < 0.05 (exploratory, uncorrected)")

In [ ]:
if 'compl' in df.columns:
    outcome = pd.to_numeric(df['compl'], errors='coerce')

    print("BINARY VARIABLES vs compl (complication rate by group)")
    print("=" * 65)
    print(f"{'Variable':<25} {'Rate if 0':>11} {'Rate if 1':>11} {'p-value':>10}")
    print("=" * 65)

    for col in binary_cat_vars:
        series = pd.to_numeric(df[col], errors='coerce')
        grp0   = outcome[series == 0].dropna()
        grp1   = outcome[series == 1].dropna()
        if len(grp0) > 5 and len(grp1) > 5:
            ct = pd.crosstab(series, outcome)
            if ct.shape == (2, 2):
                _, p, _, _ = stats.chi2_contingency(ct)
                sig = ' *' if p < 0.05 else ''
                print(f"{col:<25} {grp0.mean()*100:>10.1f}% {grp1.mean()*100:>10.1f}% {p:>10.4f}{sig}")

    print("\n* p < 0.05 (exploratory, uncorrected)")

---
## 7. Correlation Between Continuous Predictors

Pairs with |r| > 0.8 may cause multicollinearity in logistic regression.
Clinically, expect moderate correlation between FEV1% and DLCO% (both reflect lung reserve),
and between age and comorbidity count.

In [ ]:
focus_map = {
    'leeftijd':       'Age',
    'bmi':            'BMI',
    'asascore':       'ASA score',
    'ecog':           'ECOG',
    'prelongf':       'FEV1%',
    'prelongf1':      'DLCO%',
    'prelongfvo2max': 'VO2max',
    'opnduur':        'Op. duration',
    'compbloedver':   'Blood loss',
}

# Add comorbidity count
cci_cols = [c for c in PREDICTORS['Comorbidities'] if c in df.columns]
if cci_cols:
    df['_comorbidity_count'] = (
        df[cci_cols].apply(pd.to_numeric, errors='coerce').fillna(0).sum(axis=1)
    )
    focus_map['_comorbidity_count'] = 'Comorbidity count'

focus_present = {k: v for k, v in focus_map.items() if k in df.columns}
corr_df  = df[list(focus_present.keys())].apply(pd.to_numeric, errors='coerce')
corr_df.columns = list(focus_present.values())
corr_mat = corr_df.corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_mat, dtype=bool))
sns.heatmap(corr_mat, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, vmin=-1, vmax=1, ax=ax,
            annot_kws={'size': 10}, linewidths=0.5)
ax.set_title('Correlation — Key Clinical Variables', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

print("\nCorrelation pairs |r| > 0.40:")
print("-" * 55)
found_pairs = False
for i in range(len(corr_mat.columns)):
    for j in range(i):
        r = corr_mat.iloc[i, j]
        if abs(r) > 0.40:
            print(f"  {corr_mat.columns[i]:<25} <-> {corr_mat.columns[j]:<25}  r = {r:.2f}")
            found_pairs = True
if not found_pairs:
    print("  None above threshold.")

---
## 8. Predictor Readiness Summary

| Status | Meaning |
|---|---|
| Ready | < 20% missing, no major issues |
| Needs attention | 20–50% missing, OR low prevalence, OR heavy outliers |
| Exclude | > 50% missing or fundamental data problem |

In [ ]:
rows = []

for group, cols in PREDICTORS.items():
    for col in cols:
        if col not in df.columns:
            rows.append({'Group': group, 'Variable': col, 'Type': 'N/A',
                         'N present': 0, '% missing': 100.0,
                         'Notes': 'Column not found in dataset', 'Status': 'Exclude'})
            continue

        miss_pct  = df[col].isna().mean() * 100
        n_present = df[col].notna().sum()
        dtype     = ('continuous'
                     if col in CONTINUOUS
                     else 'binary/categorical')
        notes  = []
        status = 'Ready'

        if miss_pct > 50:
            status = 'Exclude'
            notes.append(f'{miss_pct:.0f}% missing')
        elif miss_pct > 20:
            status = 'Needs attention'
            notes.append(f'{miss_pct:.0f}% missing — impute')

        if dtype == 'binary/categorical':
            series  = pd.to_numeric(df[col], errors='coerce')
            n_uniq  = series.nunique(dropna=True)
            if n_uniq <= 2:
                pct_one = (series == 1).sum() / n_present * 100 if n_present > 0 else 0
                if pct_one < 2 and miss_pct <= 50:
                    if status == 'Ready': status = 'Needs attention'
                    notes.append(f'Low prevalence ({pct_one:.1f}%)')
        else:
            s = pd.to_numeric(df[col], errors='coerce').dropna()
            if len(s) > 0:
                Q1, Q3 = s.quantile(0.25), s.quantile(0.75)
                n_out  = ((s < Q1 - 1.5*(Q3-Q1)) | (s > Q3 + 1.5*(Q3-Q1))).sum()
                if n_out > len(s) * 0.05:
                    if status == 'Ready': status = 'Needs attention'
                    notes.append(f'{n_out} outliers ({n_out/len(s)*100:.1f}%)')

        rows.append({
            'Group':     group,
            'Variable':  col,
            'Type':      dtype,
            'N present': int(n_present),
            '% missing': round(miss_pct, 1),
            'Notes':     '; '.join(notes) if notes else '--',
            'Status':    status
        })

readiness = pd.DataFrame(rows)

for group in PREDICTORS.keys():
    sub = readiness[readiness['Group'] == group]
    print(f"\n{'='*70}")
    print(f"  {group.upper()}")
    print(f"{'='*70}")
    print(sub[['Variable','Type','N present','% missing','Notes','Status']].to_string(index=False))

print(f"\n{'='*40}")
print("READINESS SUMMARY")
print(f"{'='*40}")
for status, count in readiness['Status'].value_counts().items():
    print(f"  {status}: {count} variables")
print(f"\nTotal candidate predictors: {len(readiness)}")

In [ ]:
fig, ax = plt.subplots(figsize=(14, max(8, len(readiness) * 0.33)))

color_map = {'Ready': '#4CAF50', 'Needs attention': '#FF9800', 'Exclude': '#F44336'}

for i, (_, row) in enumerate(readiness.iterrows()):
    color = color_map.get(row['Status'], 'gray')
    ax.barh(i, row['N present'], color=color, alpha=0.75, edgecolor='white')
    note_txt = row['Notes'] if row['Notes'] != '--' else 'OK'
    ax.text(row['N present'] + 5, i,
            f"{row['Variable']} — {note_txt}", va='center', fontsize=8)

ax.set_yticks(range(len(readiness)))
ax.set_yticklabels(readiness['Variable'], fontsize=8)
ax.set_xlabel('N patients with data', fontsize=11)
ax.set_title('Predictor Readiness Overview — Lung Surgery', fontweight='bold', fontsize=14)

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, label=l) for l, c in color_map.items()]
ax.legend(handles=legend_elements, loc='lower right')
ax.axvline(df.shape[0] * 0.5, color='red', linestyle='--',
           linewidth=1, alpha=0.5, label='50% threshold')

plt.tight_layout()
plt.show()

---
## 9. Conclusions & Recommended Actions

Based on the analysis above, complete the table below before proceeding to preprocessing.

### Actions for the preprocessing notebook

| Action | Variables |
|---|---|
| **Compute** | `bmi` (from lengte + gewicht if not present), `leeftijd` (from gebdat + datok if not present) |
| **Median imputation** | Continuous variables with missing values (prelongf, prelongf1, prelongfvo2max, opnduur, compbloedver) |
| **Mode imputation** | Binary/categorical variables with <50% missing |
| **Sentinel cleanup** | Already done in section 1 (9 → NaN, 999 → NaN) — verify per variable |
| **One-hot encode** | procok, toegang, reggroep, zijde, roken, alcohol, prescoretnm8ct/cn/cm |
| **Ordinal encode** | asascore (1→5), ecog (0→4), sleeve (0→3), prescoreTNM stages |
| **Log-transform** | opnduur, compbloedver if heavily right-skewed |
| **Exclude** | Variables >50% missing; post-op outcomes as features; pTNM/pathology variables |
| **Low prevalence review** | Binary variables <2% positive — consider grouping or excluding |

### Sensitivity analysis (recommended)
Run the model twice:
1. **Main model**: full cohort, excluding variables with >50% missing
2. **Sensitivity model**: subset of patients with complete lung function data
   (prelongf + prelongf1 + prelongfvo2max)

If AUC improves meaningfully in the sensitivity model, this is evidence that
complete spirometry/CPET data strengthens the model.

### Next step → preprocessing notebook
Send the outputs of this notebook back and we will build the preprocessing pipeline:
1. Imputation strategy per variable
2. Encoding pipeline
3. Train/test split with stratification
4. Feature scaling where needed